# In this program we will learn the MCP (Model Context Protocol) working principle without using Actual LLM

Core Concepts of MCP:
- Client-Server Architecture
- Often compared with USB-Type-C connector which gives a single interface for various types of devices to connect. Here in Agentic-AI world, instead of using multiple API's to connect with multiple applications, we are using MCP [Connecting to DB, Executing some script/playbooks, websearch]
- MCP Servers can also be think like a restuarant who has 'Menus' [in Agentic-AI world called 'Schemas'] and 'Kitchen' [In Agentic-AI world called functions]. Where clients read the menus and places an order.


Core Components:
- Server: Light-weight program that acts like a bridge between your AI and your data (loca files , Google Drive, Company Databases, Knowledge bases) which might require different connecting methods or API calls.
- Client: AI Application wants to use those data
- Protocol: The universal set of rules that can be used to setup the bridge easily

In [45]:
# Importing the required libraries
import json
import sys
from pydantic import BaseModel, Field
from typing import Dict, Any, List

Defining the Tools which will be offered by MCP-Server

In [8]:
# Defining the Menus to offer

class AddNumbersInput(BaseModel):
    num1: float = Field(default=0.0, description="The First Number")
    num2: float = Field(default=0.0, description="The Second Number")

class DatabaseInput(BaseModel):
    user_id: int = Field(description="The unique user number")

class AnsibleInput(BaseModel):
    playbook_name: str = Field(description="Name of the playbook ended with '.yml' extension. ")
    target_servers: List[str] = Field(description="List of IP Addresses or hostnames. ")

Defining the Protocol Layer
- This helper functions mimic a real network protocol (like WebSockets or SSE).
- They serialize Python dictionaries into raw JSON strings and vice versa.

In [ ]:
# Request Wrapper into JSON-RPC 2.0 format - to be used by MCP Server
def wrap_request(msg_id: int, method: str, params: dict) -> str:
    """ Pack a client action into a standardized JSON-RPC 2.0 Text Format """
    payload = {
        "jsonrpc": "2.0",
        "id": msg_id,
        "method": method,
        "params": params
    }
    return json.dumps(payload) # convert dictionary to raw text string

In [6]:
# Wrap Successful Response  - to be used by MCP Server
def wrap_success_response(msg_id: int, result: Any) -> str:
    """ Pack a successful server result into a standard text format 
    to avoid divercified response types from different end points. """

    payload = {
        "jsonrpc": "2.0",
        "id": msg_id,
        "result": result
    }

    return json.dumps(payload)

In [7]:
# Wrap Error Response - To be used by MCP Server

def wrap_error_response(msg_id: int, code: int, message: str) -> str:
    """ Pack a server error into standard JSON-RPC text format. """
    payload = {
        "jsonrpc": "2.0",
        "id": msg_id,
        "error": f"Error Code: {code}, Message: {message}"
    }
    return json.dumps(payload)

In [41]:
# Helper function for clean output
def pretty_format_json(raw_json_str: str) -> str:
    """Parses a raw JSON string and returns a beautifully indented version."""
    try:
        data = json.loads(raw_json_str)
        return json.dumps(data, indent=4) # indent=4 makes it look clean
    except Exception:
        return raw_json_str # Fallback if it's not a JSON string

In [43]:
# Below code will help to overcome Jupyter Notebook issue
def print_full_output(text: str):
    """Forces the terminal environment to print the entire text without truncation."""
    # Split the long string into lines and print them one by one
    for line in text.splitlines():
        print(line)
    # Force the terminal to instantly output everything in the queue
    sys.stdout.flush()

Defining the MCP Server

In [33]:
class SimpleMCPServer:
    def __init__(self):
        # Defining the tools registry which will be a dictionary
        self.tools_registry = {
            "add_two_numbers": {
                "description": "Adds two given numbers. ",
                "schema": AddNumbersInput.model_json_schema(), 
                "model_class": AddNumbersInput
            },

            "get_db_user": {
                "description": "Retrieves user records from the company database",
                "schema": DatabaseInput.model_json_schema(),
                "model_class": DatabaseInput
            },
            "run_ansible_playbook": {
                "description": "Executes configuration automation across targets. ",
                "schema": AnsibleInput.model_json_schema(),
                "model_class": AnsibleInput
            }

            #'model_json_schema()' is Pydantic class member that converts class object to JSON.
        }
    
    def handle_incoming_request(self, raw_json_request: Any) -> str:
        """
        The single entry point. It only accepts and returns raw text strings.
        """
        req_id = 0 
        clean_manu = {}
        try:
            if isinstance(raw_json_request, str):
                request = json.loads(raw_json_request)
            else:
                request = raw_json_request
            #request = json.dumps(raw_json_request)

            req_id = request.get("id", 0)         # default is 0
            method = request.get("method", "")    # default blank method
            params = request.get("params", {})    # default blank dictionary

            # Route 1: Handle Tool Discovery Request
            
            if method == "tools/list":
                clean_menu = {}
                for name, t in self.tools_registry.items():
                    clean_menu[name] = {
                        "description": t["description"],
                        "schema": t["schema"]
                    }
                return wrap_success_response(req_id, clean_menu)
            
            if method == "tools/call":
                tool_name = params.get("name")
                tool_args = params.get("arguments", {})

                # Check if given tool is present into tools-registry
                if tool_name not in self.tools_registry:
                    return wrap_error_response(req_id, -32601, f"Tool : {tool_name} not found .")
                
                # When given tool is present

                tool_info = self.tools_registry[tool_name]
                ModelClass = tool_info["model_class"]
                validated_data = ModelClass.model_validate(tool_args)

                # Note: model_validate() is built-in function inside Pydantic which acts as a gatekeeper to enforce those rules on incoming data

                # Executing the core logic 
                if tool_name == "add_two_numbers":
                    return wrap_success_response(req_id, validated_data.num1 + validated_data.num2)
                elif tool_name == "get_db_user":
                    mock_db = {101: "Alice Smith (Admin)", 102: "Bob Jones (user)"}
                    return wrap_success_response(req_id, mock_db.get(validated_data.user_id))
                elif tool_name == "run_ansible_playbook":
                    output = [f"[SUCCESS] Applied {validated_data.playbook_name} to {s}" for s in validated_data.target_servers]
                    return wrap_success_response(req_id, output)
            else:
                return wrap_error_response(req_id, -32601, f"Method : {method} not found")
        except Exception as e:
            return wrap_error_response(req_id, -32602, f"Invalid request parameter : {str(e)}")

Creating MCP Client

In [47]:
class SimpleMCPClient():
    def __init__(self, server_to_connect_to):
        self.server = server_to_connect_to
        self.request_counter = 1
    
    def list_tools(self):
        """ Asks for tools using standard JSON-RPC formatting """
        msg_id = self.request_counter
        self.request_counter += 1

        # Create a standerdized protocol request string to send request to server
        raw_request = wrap_request(msg_id=msg_id, method="tools/list", params={})

        print(f"[Client -> Wire]: {raw_request}")

        # Transmit the request to server and get back the response
        print("\n" + "="*60)
        print(" 🔍 CLIENT SENDING DISCOVERY REQUEST (tools/list)")
        print("="*60)
        
        ### if you are running the code from CLI then uncomment the below line
        #print(pretty_format_json(raw_request))

        ### If you are running the code from Jupyter then use below line
        print_full_output(pretty_format_json(raw_request))

        raw_response = self.server.handle_incoming_request(raw_request)

        print("\n" + "="*60)
        print(" 📋 SERVER RESPONDED WITH AVAILABLE TOOLS")
        print("="*60)

        ### if you are running the code from CLI then uncomment the below line
        #print(pretty_format_json(raw_response)) # ✅ FIX 1: Changed raw_request to raw_response

        ### if you are running within Jupyter
        print_full_output(pretty_format_json(raw_response)) # ✅ FIX 2: Changed raw_request to raw_response

        print("="*60 + "\n")



    def call_tool(self, tool_name: str, arguments: dict):
        """ Executes a tool using standard JSON-RPC formatting. """
        msg_id = self.request_counter
        self.request_counter += 1

        # Package name and arguments for the specific tool
        params_payload = {"name": tool_name, "arguments": arguments}
        raw_request = wrap_request(msg_id=msg_id, method="tools/call", params=params_payload)
        print(f"[Client -> Wire]: {raw_request}")

        print("\n" + "#"*60)
        print(f" 🚀 CLIENT CALLING TOOL: {tool_name}")
        print("#"*60)
        print(f"Arguments Passed:\n{json.dumps(arguments, indent=4)}")


        # Transmit the request to server and get back the response
        raw_response = self.server.handle_incoming_request(raw_request)
        
        print("\n📥 SERVER EXECUTION RESPONSE:")

        ### if you are running the code from CLI then uncomment the below line
        #print(pretty_format_json(raw_response))

        ### if you are running within Jupyter
        print_full_output(pretty_format_json(raw_response))

        print("#"*60 + "\n")


Running the protocol Simulation

In [48]:
if __name__ == "__main__":
    my_mcp_server = SimpleMCPServer()
    my_mcp_client = SimpleMCPClient(my_mcp_server)

    print("========== STARTING REAL-LIFE PROTOCOL SIMULATION ==================")
    #1. Discoverying the offering by the MCP Server
    my_mcp_client.list_tools()

    #2. Executing Add Numbers tool using MCP Protocol
    my_mcp_client.call_tool("add_two_numbers", {"num1": 40, "num2": 2})

    #3. Executing Ansible Playbook using MCP Protocol
    my_mcp_client.call_tool("run_ansible_playbook", {
        "playbook_name": "restart_docker.yml",
        "target_servers": ["prod-web-1", "prod-web-2"]
    })

    # Error Handling Demonstration
    #my_mcp_client.call_tool("get_db_user", {"user_id": "NOT_A_NUMBER"})


========== STARTING REAL-LIFE PROTOCOL SIMULATION ==================
[Client -> Wire]: {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}

 🔍 CLIENT SENDING DISCOVERY REQUEST (tools/list)
{
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}

 📋 SERVER RESPONDED WITH AVAILABLE TOOLS
{
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "add_two_numbers": {
            "description": "Adds two given numbers. ",
            "schema": {
                "properties": {
                    "num1": {
                        "default": 0.0,
                        "description": "The First Number",
                        "title": "Num1",
                        "type": "number"
                    },
                    "num2": {
                        "default": 0.0,
                        "description": "The Second Number",
                        "title": "Num2",
                        "type": "number"
                    }
          